# CICA Atlas CORDEX EUR-11 stalled subset job

When: 2026-10-20

The reported request subsets `c3s-cica-atlas.r20.CORDEX-EUR-11.rcp85.mon.v25` over area `-25,34,40,72`. No time or time-component filter was supplied, so the request covers the collection's full time range.

The service reported: `Process failed: stalled-job recovery found no status update for at least 95 minutes.`

This notebook preserves the supplied request and status. The reproduction has not been run here; code cells have no recorded outputs.


## Reported WPS status

Status creation time: `2026-10-20T16:51:05Z`.

```xml
<wps:Status creationTime="2026-10-20T16:51:05Z">
    <wps:ProcessFailed>
        <wps:ExceptionReport>
            <ows:Exception exceptionCode="NoApplicableCode" locator="None">
                <ows:ExceptionText>Process failed: stalled-job recovery found no status update for at least 95 minutes.</ows:ExceptionText>
            </ows:Exception>
        </wps:ExceptionReport>
    </wps:ProcessFailed>
</wps:Status>
```


## Original WPS workflow

The decoded JSON below preserves the supplied `ComplexData` CDATA payload, including the empty input name, `inputs/` collection reference, and `subset__1` step name.


In [1]:
request = {
    "inputs": {"": ["c3s-cica-atlas.r20.CORDEX-EUR-11.rcp85.mon.v25"]},
    "steps": {
        "subset__1": {
            "run": "subset",
            "in": {"collection": "inputs/", "area": "-25,34,40,72"},
        }
    },
    "outputs": {"output": "subset__1/output"},
    "doc": "workflow",
}

request


{'inputs': {'': ['c3s-cica-atlas.r20.CORDEX-EUR-11.rcp85.mon.v25']},
 'steps': {'subset__1': {'run': 'subset',
   'in': {'collection': 'inputs/', 'area': '-25,34,40,72'}}},
 'outputs': {'output': 'subset__1/output'},
 'doc': 'workflow'}

## Run the two workflows

Run the cells in order: first without an area, then with the original area. Both use the full time range. Sizes and URLs come from Metalink metadata; the result files are not downloaded.


In [2]:
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


## 1. Without area


In [3]:
ds = ops.Input(
    "ds", ["c3s-cica-atlas.r20.CORDEX-EUR-11.rcp85.mon.v25"]
)

wf = ops.Subset(ds)

resp = wf.orchestrate()
resp.ok


True

In [4]:
print(f"Succeeded: {resp.ok}")
print(f"Status: {resp.status}")
if resp.ok:
    print(f"Output size: {resp.size:,} bytes ({resp.size_in_mb:.2f} MiB)")
    for url in resp.download_urls():
        print(url)


Succeeded: True
Status: ProcessSucceeded
Output size: 0 bytes (0.00 MiB)
https://data.mips.climate.copernicus.eu/thredds/fileServer/esg_c3s-cica-atlas/v025/CORDEX-EUR-11/rcp85/r20_CORDEX-EUR-11_rcp85_mon_200601-210012_v025.nc


## 2. With area


In [5]:
ds = ops.Input(
    "ds", ["c3s-cica-atlas.r20.CORDEX-EUR-11.rcp85.mon.v25"]
)

wf = ops.Subset(ds, area="-25,34,40,72")

resp = wf.orchestrate()
resp.ok


True

In [6]:
print(f"Succeeded: {resp.ok}")
print(f"Status: {resp.status}")
if resp.ok:
    print(f"Output size: {resp.size:,} bytes ({resp.size_in_mb:.2f} MiB)")
    for url in resp.download_urls():
        print(url)


Succeeded: True
Status: ProcessSucceeded
Output size: 9,527,454,095 bytes (9086.09 MiB)
http://rook7.cloud.dkrz.de:80/outputs/rook/21a8cbb6-cd7a-11f1-a686-fa163eb671ca/r20_CORDEX-EUR-11_rcp85_mon_20060101-20090701.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/21a8de62-cd7a-11f1-a686-fa163eb671ca/r20_CORDEX-EUR-11_rcp85_mon_20090801-20130201.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/21a8eab0-cd7a-11f1-a686-fa163eb671ca/r20_CORDEX-EUR-11_rcp85_mon_20130301-20160901.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/21a8f636-cd7a-11f1-a686-fa163eb671ca/r20_CORDEX-EUR-11_rcp85_mon_20161001-20200401.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/21a9054a-cd7a-11f1-a686-fa163eb671ca/r20_CORDEX-EUR-11_rcp85_mon_20200501-20231101.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/21a91076-cd7a-11f1-a686-fa163eb671ca/r20_CORDEX-EUR-11_rcp85_mon_20231201-20270601.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/21a91a8a-cd7a-11f1-a686-fa163eb671ca/r20_CORDEX-EUR-11_rcp85_mon_20270701-20310101.nc
http://ro

## Investigation notes

The status records stalled-job recovery after at least 95 minutes without a status update. It does not establish whether processing stopped, a worker failed, or status reporting stopped. No server traceback, job identifier, or input-file inspection was supplied.

Capture the job identifier, worker logs, resource usage, and last status update when investigating. Inspect the files resolved for the exact collection and version above. Compare the unchanged original request with shorter time ranges to assess whether the full time range affects runtime; keep any diagnostic changes separate from the preserved request.
